# Lesson 7: The rotation engine, comparing rotations honestly

So far FieldSense *describes* a field: its history (the barcode), its soil cover, its climate odds for 2026. Today it starts to **help decide**: *"Which rotation suits this field, and how sure are we?"*

We compare four rotations, including the one each plot follows now:

| Key | Rotation | Plots now |
|---|---|---|
| `CS` | Corn – soybean | 1, 2 (simple) |
| `CS+rye` | Corn – soybean, with cereal rye sown after soybean | |
| `SWC` | Soybean – winter wheat – corn | |
| `SWC+clover` | Soybean – winter wheat (red clover sown into it) – corn | 3, 4 (diverse) |

For each one we compute **soil cover**, **exposure to 2026 climate risks**, and **nutrient and pest facts**, each with its evidence label. Then we ask the hardest question honestly: *which one is "best"?*

**Job skills in this lesson**
- *Software engineering:* keeping agronomy in a **data file** (not in code), `dataclass`es, and **property tests** (a new kind of test)
- *Data science:* **validating a model against measurements**, **multi-criteria decisions**, and **robustness analysis** (Monte Carlo over priorities instead of over weather)
- *Product:* a **trade-off table, not one magic score**, and explanations generated from the numbers, so the words can't say more than the data

**Time:** about 2 hours. There's no downloading: everything comes from Lessons 4–6 files in your Drive.


## Step 0: Review and merge pull request #2 (on GitHub)

Open https://github.com/SetarehAlamdar/FieldSense/pull/2 and go to **Files changed**. This time, review it like a colleague would. Here's what to look at, and a question to ask yourself for each:

1. **`src/fieldsense/data/crops.json`**, the crop table. *Is a JSON file the right home for these numbers? Who should be able to edit them?*
2. **`src/fieldsense/rotation.py`**: read the docstring at the top first. It lists the four design choices. *Do you agree with each one? Which would you challenge?*
3. **`tests/test_rotation.py`**: read the docstring, which explains the difference between **example tests** and **property tests**. Find one of each.
4. **`src/fieldsense/climate.py`**: a new risk (`short_fall_after_soybean`) replaces the one that never happened at Elora. Notice the old one is *kept* and marked **DEPRECATED** rather than deleted. *Why not just delete it?* (Hint: Lesson 6, Step 10.)

**Leave at least one review comment.** Click the **+** next to any line and ask a question or suggest a change. Real reviews always have comments, and you'll get practice writing them. Then **Merge pull request**.


## Project setup: run this first, every session

Everything we save, **data files and code modules**, goes into your **Google Drive** under `MyDrive/fieldsense/`. Colab wipes its own disk whenever a session ends, but Drive keeps everything, so each lesson can pick up the previous lessons' files.

Colab will ask permission to access your Drive the first time each session; click **Allow**.


In [ ]:
# ---- Project setup: run this first, every session ----
import os, sys, glob, shutil
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/fieldsense"
except ImportError:                       # running outside Colab: use a local folder instead
    PROJECT_DIR = os.path.abspath("fieldsense")
CACHE_DIR = os.path.join(PROJECT_DIR, "data_cache")
os.makedirs(CACHE_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)                     # %%writefile now saves code modules into Drive too
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)       # lets `import fieldsense_data` find them

# One-time rescue: copy anything still sitting in Colab's temporary folder into Drive
for f in glob.glob("/content/data_cache/*"):
    if not os.path.exists(os.path.join(CACHE_DIR, os.path.basename(f))):
        shutil.copy(f, CACHE_DIR)
        print("Rescued into Drive:", os.path.basename(f))

print("Project folder:", PROJECT_DIR)
print("Saved data files:", sorted(os.listdir(CACHE_DIR)) or "none yet")

PRIVATE_DIR = os.path.join(PROJECT_DIR, "private")       # lab records: NEVER goes to GitHub
PROFILE_DIR = os.path.join(PROJECT_DIR, "profiles")      # field profiles + evidence registry
DEMO_DIR = os.path.join(PROJECT_DIR, "demo")             # figures for slides and website
for d in (PRIVATE_DIR, PROFILE_DIR, DEMO_DIR):
    os.makedirs(d, exist_ok=True)

## Step 1: Install the latest code and run the tests

You should see **version 0.3.0** and **44 passed**. (If you see 0.2.0: merge the PR, then **Runtime → Restart session** and run from the top.)


In [ ]:
import subprocess
REPO_URL = "https://github.com/SetarehAlamdar/FieldSense.git"
REPO_DIR = os.environ.get("FIELDSENSE_REPO", "/content/FieldSense")

def sh(cmd):
    """Run a shell command; print its output; stop the notebook if it fails."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print((r.stdout + r.stderr).strip()[-1500:])
    if r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}")

url = REPO_URL
try:                                           # private repo: use a token from Colab Secrets
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{token}@")
except Exception:
    pass                                       # public repo (or not in Colab): no token needed

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    sh(f"git -C {REPO_DIR} pull --ff-only")
else:
    sh(f"git clone --quiet {url} {REPO_DIR}")
sh(f'pip install -q -e "{REPO_DIR}[dev]"')

SRC = os.path.join(REPO_DIR, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import fieldsense as fs
print("fieldsense version", fs.__version__, "from", os.path.dirname(fs.__file__))
sh(f"git -C {REPO_DIR} log --oneline -5")
sh(f"cd {REPO_DIR} && python -m pytest -q")

## Step 2: Rebuild the 2026 futures, and check reproducibility

The new risk, *too little fall warmth for a cover crop sown after soybean*, needs a new indicator that the Lesson 6 futures don't have. So we rebuild them **with exactly the same settings and the same seed (2026)**.

That's also a **reproducibility test**: with the same code, data, settings and seed, the futures should match Lesson 6. The cell checks three things: are the random draws the same, how different are the temperature shifts, and did any risk count change?

**Expect a tiny difference, and learn from it.** Lesson 6 saved `temp_sd` to the config **rounded to 2 decimals** (e.g. 0.17), but built its futures with the full-precision number (e.g. 0.1693...). Same seed, slightly different input → temperature shifts differ by about 0.01 °C, and a handful of futures right on a threshold may flip. The random *draws* (which years, which rain factors) should still be identical.

This is one of the most common reproducibility bugs in real ML systems: **a parameter is saved in a different precision (or format) from the one actually used.** The rule: save exactly what you used, at full precision, and round only when *displaying*. From now on, the config holds the value the futures are built from, so future runs will match exactly.

The warming trend wasn't saved in Lesson 6 (a small gap, now fixed: we save it to the config). Recomputing it is safe, though, because its bootstrap also uses a fixed seed.

At the end, `uninformative_risks` lists any risk that (almost) never or (almost) always happens. A risk like that can't help choose between options. This is the check you did by eye in Lesson 6 when you saw "0 in 100", now automated.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from fieldsense import rotation as rt
from fieldsense.profile import climate_odds, load_profile, save_profile, check_profile, SCHEMA_VERSION
from fieldsense import EvidenceRegistry, Evidence

config = json.load(open(os.path.join(CACHE_DIR, "fieldsense_config.json")))
w2 = pd.read_csv(os.path.join(CACHE_DIR, config["temp_correction"]["weather_file"]), index_col=0, parse_dates=True)
w2 = w2.loc["1996":"2025"]

annual = w2["T2M"].groupby(w2.index.year).mean()
trend = fs.warming_trend(annual)
TEMP_TREND = trend["slope_per_year"] if trend["low"] > 0 else 0.0
TEMP_SD = config["temp_correction"]["yearly_residual_sd_degC"]
RAIN_SD = config["precip_correction"].get("rain_uncertainty_fraction", 0.10)
config["climate_futures"] = {"trend_degC_per_year": round(TEMP_TREND, 4), "trend_ci90": [round(trend["low"], 4), round(trend["high"], 4)],
                             "temp_sd": TEMP_SD, "rain_sd": RAIN_SD, "n": 1000, "seed": 2026, "target_year": 2026}
json.dump(config, open(os.path.join(CACHE_DIR, "fieldsense_config.json"), "w"), indent=2)

futures = fs.make_futures(w2, n=1000, temp_trend=TEMP_TREND, rain_sd=RAIN_SD, temp_sd=TEMP_SD, seed=2026)
hist = fs.indicators_by_year(w2)

old_file = os.path.join(CACHE_DIR, "climate_futures_elora_2026.csv")
if os.path.exists(old_file):
    old = pd.read_csv(old_file, index_col=0)
    same_draws = (old["source_year"].values == futures["source_year"].values).all() and                  np.allclose(old["rain_factor"], futures["rain_factor"])
    shift_diff = float(np.abs(old["temp_shift"] - futures["temp_shift"]).max())
    print("Reproducibility check against Lesson 6:")
    print("  same random draws (years picked, rain factors):", "YES" if same_draws else "NO, investigate!")
    print(f"  largest temperature-shift difference: {shift_diff:.4f} degC")
    old_risks = {k: v for k, v in fs.RISKS.items() if v[0] in old.columns}   # only risks the old file can measure
    before, after = fs.risk_counts(old, old_risks), fs.risk_counts(futures, old_risks)
    changed = {k: (before[k]["count"], after[k]["count"]) for k in old_risks}
    changed = {k: v for k, v in changed.items() if v[0] != v[1]}
    print("  risk counts that changed (Lesson 6 -> now, out of 1000):", changed or "none")
futures.to_csv(old_file)

odds_evidence = ["power_precip_corrected", "power_temperature", "assumed_risk_thresholds"]
odds = climate_odds(futures, hist, evidence=odds_evidence, target_year=2026)
for k, r in odds["risks"].items():
    print(f"{round(100*r['share']):3d} in 100   (past {r['past_years']:>8s})   {r['text']}")
print("\nUninformative risks (never or always happen):", fs.uninformative_risks(fs.risk_counts(futures)))

**Check:** the new `short_fall_after_soybean` risk should land somewhere between "never" and "always". If it shows up in the uninformative list, its threshold (150 degree-days) needs adjusting for Elora. Tell me, and we'll treat it the way we treated the wheat rule in Lesson 4.

The deprecated risk will probably still be listed. That's expected: it's kept only so older readers don't break.


## Step 3: The crop table, agronomy as data

Open the table the engine uses. Each crop has:

| Field | Meaning |
|---|---|
| `green` | Days of the year (from–to) when the crop is clearly green, NDVI ≥ 0.4: the same threshold as the barcode |
| `green_previous_fall` | Winter wheat is sown in October, so it's green the fall **before** its year |
| `family`, `legume` | Plant family; legumes (soybean, clover) fix their own nitrogen |
| `sensitive_to` | Which Lesson 6 risks can damage this crop |
| covers: `green_fall`, `green_next_spring` | When a cover crop is green after it's sown, and the next spring before it's ended |

**Why a JSON file and not Python code?** Three reasons:
1. **The right person can edit it.** An agronomist can correct "corn is green from day 165" without reading or touching Python.
2. **One change, everywhere.** The API, the website and the tests all read the same file.
3. **It's honest about its status.** The file says at the top that the values are rules of thumb (**Assumed**) to check against OMAFRA's guide.

**Try it (optional):** `rt.load_crop_table("my_crops.json")` loads an edited copy. Change one window, rerun Steps 4–7, and watch what moves. That's a quick *sensitivity check* on an assumption.


In [ ]:
table = rt.load_crop_table()
print(table["_about"], "\n")
rows = [{"crop": k, "green (day of year)": v["green"], "fall before": v.get("green_previous_fall", ""),
         "legume": v["legume"], "sensitive to": ", ".join(v["sensitive_to"]) or "-"} for k, v in table["crops"].items()]
rows += [{"crop": k + " (cover)", "green (day of year)": v["green_fall"], "fall before": "",
          "legume": v["legume"], "sensitive to": v.get("needs_fall_warmth", "-")} for k, v in table["covers"].items()]
display(pd.DataFrame(rows).set_index("crop"))

## Step 4: Draw each rotation's calendar, the "modelled barcode"

`rt.Rotation` is a **dataclass**: a small, named container for related values (`key`, `name`, `crops`, `covers`). It's like a row in a table that Python understands, which is cleaner than passing loose lists around and easy to test.

`green_calendar` paints, for each year of the rotation, the days when *something* is green: the main crop, wheat's fall start, a cover crop's fall growth and its green spring after. The rotation **repeats**, so "the fall before year 1" is the last year of the cycle. That's why the code uses `% n` (the remainder after dividing by the cycle length), which wraps the years around.

The picture uses the same colours as your real barcode, **green = growing, brown = bare**, so you can put them side by side.


In [ ]:
from matplotlib.colors import ListedColormap
fig, axes = plt.subplots(len(rt.CANDIDATES), 1, figsize=(11, 1.3 * sum(len(r) for r in rt.CANDIDATES) / 2 + 2))
for ax, r in zip(axes, rt.CANDIDATES):
    g = rt.green_calendar(r, table)
    ax.imshow(g, aspect="auto", cmap=ListedColormap(["#b98a55", "#3f8f3a"]), interpolation="nearest",
              extent=[0.5, 366.5, len(r) - 0.5, -0.5])
    labels = [c.replace("_", " ") + (" + " + r.covers[i].replace("_", " ") if i in r.covers else "") for i, c in enumerate(r.crops)]
    ax.set_yticks(range(len(r))); ax.set_yticklabels(labels, fontsize=8)
    ax.set_xticks([1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]); ax.set_xticklabels("J F M A M J J A S O N D".split())
    ax.set_title(f"{r.key}: {r.name}", loc="left", fontsize=10)
    for s in ax.spines.values(): s.set_visible(False)
plt.tight_layout(); plt.savefig(os.path.join(DEMO_DIR, "rotation_calendars.png"), dpi=200); plt.show()

## Step 5: Measured beats modelled, validating the calendar model

The calendar is a **model**: it guesses soil cover from rules of thumb. For two of the four rotations, though, we have **measurements**. Plots 1–2 grow `CS` and plots 3–4 grow `SWC+clover`, and their real greenness since 2018 is in your HLS data (Lesson 4).

So we compare the model with the measurement on three numbers, computed **the same way** for both (NDVI ≥ 0.4 counts as green, using only the days the satellite actually saw):
- share of the season (1 Mar – 30 Nov) that's green
- share of the off-season that's green
- share of spring (15 Apr – 15 Jun) that's bare

**Pre-registered rule (decided now, before seeing the result):** the model counts as **Validated** if every difference is within **10 percentage points** for both rotations. Otherwise it stays **Modelled**, and the differences become its stated error.

**Why this matters (an ML-engineering idea):** any simulator, whether a crop calendar, a physics model or a digital twin, must be checked against reality wherever reality is available. And where you *have* a measurement, you **use the measurement**, not the model. So for each plot's *current* rotation, the app will show the measured cover, and the model is used only for rotations the field hasn't tried.


In [ ]:
from fieldsense.phenology import daily_ndvi, barcode_matrix
hls = pd.read_csv(os.path.join(CACHE_DIR, "hls_plot_ndvi.csv"), parse_dates=["date"])
hls = hls[hls["ndvi"].notna()]
doy = np.arange(1, 366)
season = (doy >= 60) & (doy <= 334)
off = season & ~((doy >= 135) & (doy <= 273))
spring = (doy >= 105) & (doy <= 166)

def measured_cover(plot, since=2018):
    m = barcode_matrix(daily_ndvi(hls[hls["plot"] == plot][["date", "ndvi"]]))
    m = m[m.index >= since].values
    seen = lambda mask: np.isfinite(m[:, mask])
    green = lambda mask: (m[:, mask] >= 0.4) & seen(mask)
    return {"green_share_season": green(season).sum() / seen(season).sum(),
            "green_share_off_season": green(off).sum() / seen(off).sum(),
            "spring_bare_share": 1 - green(spring).sum() / seen(spring).sum()}

GROUPS = {"CS": [1, 2], "SWC+clover": [3, 4]}
cand = {r.key: r for r in rt.CANDIDATES}
rows, MEASURED = [], {}
for key, plots in GROUPS.items():
    meas = pd.DataFrame([measured_cover(p) for p in plots]).mean().to_dict()
    MEASURED[key] = meas
    mod = rt.cover_metrics(cand[key], table)
    for metric in meas:
        rows.append({"rotation": key, "metric": metric, "measured": meas[metric], "modelled": mod[metric],
                     "difference": mod[metric] - meas[metric]})
val = pd.DataFrame(rows).set_index(["rotation", "metric"])
display((100 * val).round(0).astype(int).astype(str) + " %")

MAX_DIFF = val["difference"].abs().max()
MODEL_LABEL = "validated" if MAX_DIFF <= 0.10 else "modelled"
print(f"Largest difference: {100*MAX_DIFF:.0f} percentage points -> calendar model is {MODEL_LABEL.upper()}")

**Read the differences carefully, whichever way the verdict went:**
- If the model shows **more** green than the satellite, the windows in `crops.json` may be too generous. For example, red clover may take longer to green up after wheat harvest than day 227, or not reach NDVI 0.4 in a dry August.
- If it shows **less**, there may be weeds or volunteer plants, or crops greening earlier than the rule says.
- Cloud gaps don't bias this comparison, because we compare *shares of observed days* on both sides.

If the model is off by more than 10 points, **don't tune it until it matches.** Tuning on the same data you validate on is the Lesson 4 trap: it would pass the check and prove nothing. Instead, note the error honestly, and remember that it only affects rotations the field hasn't tried. For a later lesson, the clean fix is to fit the windows on some plots or years and test on others.


In [ ]:
reg = EvidenceRegistry.from_json(os.path.join(PROFILE_DIR, "evidence.json"))
desc = "; ".join(f"{k} {m.replace('_', ' ')}: {100*d:+.0f} pts" for (k, m), d in val["difference"].items())
if MODEL_LABEL == "validated":
    reg.add(Evidence("rotation_calendar_model", "validated",
        claim="Soil cover for rotations, estimated from a crop calendar (fieldsense/data/crops.json).",
        source="Crop calendar rules of thumb, checked against HLS greenness",
        test="Modelled vs satellite-measured cover for CS (plots 1-2) and SWC+clover (plots 3-4), 2018-2025; rule: all within 10 points",
        result=desc, caveat="Checked on two rotations at one site; calendars may differ elsewhere."))
else:
    reg.add(Evidence("rotation_calendar_model", "modelled",
        claim="Soil cover for rotations, estimated from a crop calendar (fieldsense/data/crops.json).",
        source="Crop calendar rules of thumb",
        caveat=f"Differs from satellite measurement by up to {100*MAX_DIFF:.0f} points ({desc})."))
reg.to_json(os.path.join(PROFILE_DIR, "evidence.json"))
print("Saved evidence:", reg.get("rotation_calendar_model").label)

## Step 6: The trade-off table

`rt.compare` puts every rotation and every metric in one table:

| Metric | Means | Better when |
|---|---|---|
| `green_share_off_season` | Living cover outside summer | higher |
| `spring_bare_share` | Bare soil when spring storms hit | lower |
| `climate_risk_total` | Risk-hits per 100 crop-years in 2026 (see below) | lower |
| `legume_share` | Nitrogen-fixing crop or cover per year | higher |
| `min_years_between_same_crop` | Break for pests and diseases (e.g. corn rootworm, soybean cyst nematode) | higher |
| `crops_to_manage` | Different crops and covers: equipment, seed, markets, timing | **lower (a cost)** |

**How climate exposure is calculated, in one line:** for each year of the rotation, *P(the risk happens in 2026) × (is that year's crop sensitive to it?)*, averaged over the cycle. For example, pollination heat is 21 in 100, corn is sensitive, and corn is 1 year in 2 in `CS` → **10.5 per 100 crop-years**. In `SWC` corn is 1 year in 3 → **7 per 100**. For heavy rain we use the *share of spring that's bare* instead of yes/no, which is where cover crops help.


In [ ]:
comparison = rt.compare(rt.CANDIDATES, odds, table)
show = ["green_share_off_season", "spring_bare_share", "climate_risk_total", "legume_share",
        "min_years_between_same_crop", "crops_to_manage"]
pretty = comparison[show].copy()
for c in ["green_share_off_season", "spring_bare_share", "legume_share"]:
    pretty[c] = (100 * pretty[c]).round(0).astype(int).astype(str) + " %"
display(pretty)
print("\nClimate risk breakdown (hits per 100 crop-years):")
display(comparison[[c for c in comparison.columns if c.startswith("risk_")]].round(1))

## Step 7: Which is "best"? It depends on what you value, so we measure that too

To rank the rotations we'd have to **weight** the criteria. Is soil cover twice as important as simplicity? Half as important? **That's not a fact. It's a farmer's values.** A tool that hides one fixed set of weights inside a single score is quietly making that choice for them.

So we do two things.

**A. Show rankings for a few example farmers.** Each one is just a different set of weights.

**B. A robustness check: try every possible set of weights.** `rank_robustness` draws 2,000 random weightings, ranks the rotations under each, and counts how often each one comes out best.
- The weights come from a **Dirichlet distribution**. That sounds fancy, but it just means: *randomly split 100% of "importance" across the six criteria, where every possible split is equally likely.*
- If a rotation wins under 90% of weightings, the choice barely depends on priorities: it's **robust**. If wins are split, the decision really is about values, and the app should ask the farmer.

This is the same Monte Carlo idea as Lesson 6, applied to **priorities** instead of weather.

**The honest caveat, which you should say in the demo:** we have no data on **yield, profit or labour**, so they aren't criteria. Leaving out costs makes diverse rotations look better than many farmers would find them. I only noticed this when I first ran the check without `crops_to_manage`: one rotation won under **100%** of weightings. A result that perfect is usually a sign of a missing criterion, not a perfect option.


In [ ]:
FARMERS = {
    "soil-health first": {"green_share_off_season": 3, "spring_bare_share": 3, "climate_risk_total": 1,
                          "legume_share": 1, "min_years_between_same_crop": 1, "crops_to_manage": 1},
    "keep it simple":    {"green_share_off_season": 1, "spring_bare_share": 1, "climate_risk_total": 1,
                          "legume_share": 0, "min_years_between_same_crop": 1, "crops_to_manage": 5},
    "balanced":          {k: 1 for k in rt.CRITERIA},
}
for who, w in FARMERS.items():
    r = rt.weighted_ranking(comparison, w)
    print(f"{who:18s} -> " + "  >  ".join(f"{k} ({v:.2f})" for k, v in r.items()))

robust = rt.rank_robustness(comparison, n=2000, seed=0)
print("\nBest under this share of 2,000 random weightings:")
print((100 * robust).round(1).astype(str) + " %")
ax = (100 * robust).plot.barh(color="#3f8f3a", figsize=(7, 2.6))
ax.invert_yaxis(); ax.set_xlabel("% of possible priority mixes where this rotation is best")
ax.set_title("How robust is the choice?", loc="left"); plt.tight_layout()
plt.savefig(os.path.join(DEMO_DIR, "rotation_robustness.png"), dpi=200); plt.show()

## Step 8: Each plot's options, explained in five parts

For each plot:
1. **What it does now:** `detect_current` reads the satellite history. Wheat in recent years means a 3-year rotation; green off-season means cover crops.
2. **What comes in 2026:** `next_crop` lines the cycle up with the wheat years. For corn–soybean the satellite can't tell corn from soybean (Lesson 4), so it says so and asks the farmer instead of guessing.
3. **Five-part explanation**, for the current rotation and the most robust alternative:

| Part | Evidence |
|---|---|
| **What**: the rotation | Validated (the satellite crop history was checked against field records) |
| **Soil cover** | **Measured** for the current rotation; the calendar model's label for others |
| **Climate 2026** | Weakest of rain, temperature and thresholds → **Assumed** |
| **Nutrients & pests** | Assumed (agronomic rules) |
| **Unknowns** | What we didn't estimate: always shown |

The sentences are **generated from the numbers by templates**, not written freely. So the explanation can never claim more than the data shows. (In week 13 of the bootcamp, an AI assistant will *read* these explanations, and that's exactly why they need to be trustworthy first.)


In [ ]:
profile_files = sorted(f for f in os.listdir(PROFILE_DIR) if f.startswith("plot_") and f.endswith(".json"))
crop_hist_label = reg.get("hls_crop_history").label
odds_label = reg.weakest(odds_evidence)
best_overall = robust.index[0]
OPTIONS = {}

for fname in profile_files:
    p = load_profile(os.path.join(PROFILE_DIR, fname))
    current = rt.detect_current(p["history"])
    plot_no = int(fname.split("_")[1].split(".")[0])
    measured = measured_cover(plot_no)
    alternative = best_overall if best_overall != current else robust.index[1]
    explanations = {}
    for key in (current, alternative):
        is_current = key == current
        explanations[key] = rt.explain(
            comparison.loc[key], current_key=current,
            evidence={"what": crop_hist_label if is_current else "assumed",
                      "soil_cover": "measured" if is_current else MODEL_LABEL,
                      "climate_2026": odds_label, "nutrients_pests": "assumed"},
            measured_cover=measured if is_current else None)
    OPTIONS[fname] = {"current": current, "next_crop_2026": rt.next_crop(current, p["history"]),
                      "alternative": alternative, "explanations": explanations}

for fname in (profile_files[0], profile_files[-1]):
    o = OPTIONS[fname]
    print(f"===== {fname}: now {o['current']}; in 2026: {o['next_crop_2026']} =====")
    for key, ex in o["explanations"].items():
        print(f"--- {key} {'(current)' if key == o['current'] else '(alternative)'}")
        for part, d in ex.items():
            print(f"  [{d['evidence'].upper():9s}] {part:16s} {d['text']}")
    print()

**Read one explanation out loud as if to a farmer.** Does every sentence say something true and useful? Is anything missing that a farmer would ask about? Write your answer down: it's product feedback for Lesson 9 (the demo page).


## Step 9: Save to the profiles (schema 0.3)

We add a `rotation_options` section to each plot's profile: the comparison table, the robustness shares, the current rotation, the 2026 crop and the explanations. We also refresh `climate_odds` so it includes the new risk.

Only additions, nothing renamed or removed: a **backward-compatible** change, so the version goes **0.2 → 0.3**.


In [ ]:
for fname in profile_files:
    path = os.path.join(PROFILE_DIR, fname)
    p = load_profile(path)
    o = OPTIONS[fname]
    p["schema"] = SCHEMA_VERSION
    p["climate_odds"] = odds
    p["rotation_options"] = {
        "current": o["current"],
        "next_crop_2026": o["next_crop_2026"],
        "candidates": json.loads(comparison.round(4).reset_index().to_json(orient="records")),
        "criteria": {k: ("higher is better" if v > 0 else "lower is better") for k, v in rt.CRITERIA.items()},
        "robustness_best_share": {k: round(float(v), 3) for k, v in robust.items()},
        "not_included": ["yield", "profit", "labour"],
        "calendar_model_evidence": MODEL_LABEL,
        "explanations": o["explanations"],
    }
    problems = check_profile(p)
    save_profile(p, path)
    print(f"{fname}: schema {p['schema']}, current {o['current']}, 2026: {o['next_crop_2026']} -> {'OK' if not problems else problems}")

## Summary

Run this and paste the output back to me.


In [ ]:
print("=== Lesson 7 summary ===")
print("fieldsense", fs.__version__); sh(f"git -C {REPO_DIR} log --oneline -1")
print("New risk:", {k: odds["risks"][k]["futures"] for k in ("short_fall_after_soybean",)},
      "| uninformative:", fs.uninformative_risks(fs.risk_counts(futures)))
print("\nCalendar model vs satellite (percentage points):")
print((100 * val["difference"]).round(0).to_string())
print("Calendar model evidence:", MODEL_LABEL)
print("\nTrade-off table:")
print(comparison[show].round(2).to_string())
print("\nRobustness (best under % of weightings):", {k: round(100 * v, 1) for k, v in robust.items()})
for who, w in FARMERS.items():
    print(f"  {who}: {rt.weighted_ranking(comparison, w).index[0]}")
print("\nPer plot:", {f: (o["current"], o["next_crop_2026"]) for f, o in OPTIONS.items()})

## What you learned

**Software engineering**
- **Configuration as data**: agronomy lives in `crops.json`, so the right person can edit it, and every part of the system reads the same values.
- **Dataclasses**: small, named containers that make code clearer and easier to test.
- **Property tests** check rules that must always hold ("adding a cover never reduces cover"), so they're less tied to our own assumptions than example tests.
- **Deprecation**: mark something old as deprecated, keep it working, and remove it at the next major version.

**Data science**
- **Validate a model against measurements**, with the rule decided in advance, and *use the measurement* wherever you have one.
- **Don't tune on your validation data** (the Lesson 4 trap, again).
- **Multi-criteria decisions**: weights are values, not facts.
- **Robustness analysis**: Monte Carlo over priorities. A 100% win is a warning sign of missing criteria.

**Product**
- Show a **trade-off table**, not one magic score.
- **Generate explanations from numbers**, and always list the unknowns.

---

## Interview practice (answer out loud first, then open the model answer)

**Q1. "Your recommendation system recommends the same option to every user. What might be wrong?"**
<details><summary>Model answer</summary>

Possibly nothing: one option may truly dominate. But first check the objective. Often it's **missing criteria**, especially costs, so one option looks better on everything measured. Other causes: weights that don't reflect users' differing preferences; features that don't vary between users (uninformative); or a bug in normalisation. I'd check robustness across weightings, add the missing cost criteria (or explicitly say they're excluded), and validate against what users actually choose. In FieldSense, adding just one cost (crops to manage) changed a 100% win into a 91% one and revealed a real trade-off.
</details>

**Q2. "How do you validate a simulation model when you can't run controlled experiments?"**
<details><summary>Model answer</summary>

Compare its outputs to real observations wherever they exist, using the same definitions on both sides, with an acceptance threshold set **before** looking. Don't calibrate and validate on the same data: fit on some sites or years, test on others. Report the error as the model's uncertainty. Where a direct measurement exists, use it instead of the model, and use the model only to extrapolate (e.g. to options not yet tried), stating that its error was measured on the observed cases.
</details>

**Q3. "What's a property-based test, and when is it better than an example test?"**
<details><summary>Model answer</summary>

An example test checks a specific input against a known output. A property test checks a rule that must hold for all inputs: monotonicity ("adding a cover never reduces cover"), bounds ("shares are between 0 and 1"), invariants ("same seed → same result"). It's better when the exact right answer is unknown or depends on assumptions, and it catches edge cases you didn't think of. Libraries like Hypothesis generate many random inputs automatically.
</details>

**Q4. "Why would you keep parameters like a crop calendar in a config file rather than in code?"**
<details><summary>Model answer</summary>

Separation of concerns: domain experts can review and change values without editing code; one source of truth for every service (API, batch jobs, tests); changes are visible and reviewable in version control; you can swap configurations for different regions without code changes. The trade-offs: you need validation of the config (schema checks) and tests that load the real file, so a typo can't silently break production.
</details>

**Next, Lesson 8: the API.** You'll build the "waiter" from our restaurant analogy: a FastAPI service that serves these profiles and rotation options to the website, with your tests running automatically on GitHub every time you push.
